In [1]:
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqIO import parse
import re
import os

In [2]:
import Bio
print(Bio.__version__)


1.84


In [2]:
!ls

0-get-real-sequences-from-cohordinates.ipynb
1_premature_stopcodon_identification.ipynb
bAmy-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB118-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_PCB118-vs-CTR
bAmy-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_PCB153-vs-CTR
CTR-Tau-vs-CTR
CTR-Tau-vs-CTR_AND_PCB118-vs-CTR
CTR-Tau-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
CTR-Tau-vs-CTR_AND_PCB153-vs-CTR
DASE_genes_enrichment.ipynb
DEGS_enrichment.ipynb
DEGS_RBP.ipynb
Figure1.ipynb
Goatools.ipynb
mirna-codes
PCB118-vs-CTR
PCB118-vs-CTR_AND_PCB153-vs-CTR
PCB153-vs-CTR


In [3]:
!ls PCB153-vs-CTR/DASE

de_novo_annotated_DASE_PCB153-vs-CTR.txt


In [4]:
path='PCB153-vs-CTR/DASE/de_novo_annotated_DASE_PCB153-vs-CTR.txt'

In [5]:
path2='/home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.fa'

In [6]:
DASE_GENE=['ABCD4', 'KMT2A', 'DCTN1', 'HTRA2', 'VLDLR', 'AFF4', 'OPA1', 'FLAD1', 'NSD1', 'KMT5B', 'ACADM', 'TXNL4A', 'CTSA', 'RUBCN', 'ADSL', 'STRADA', 'UROS', 'TINF2', 'LAMB1', 'GDAP1', 'POMGNT1', 'CLK2', 'DCX', 'RAB3GAP2', 'TAF1', 'FGFR1']
#genes implicated in intellectual disability only (resulted enriched by disgenet ) on dase genes in comparison PCB153-vs-ctr

In [7]:
ri_df_sig=pd.read_csv(path,sep='\t',usecols=['Class', 'GeneID', 'geneSymbol', 'chr', 'strand', 'coho4','coho5'])
ri_df_sig['geneSymbol']=ri_df_sig['geneSymbol'].fillna(ri_df_sig['GeneID'])
df_sig_ri=ri_df_sig[ri_df_sig['Class']=='RI']

In [8]:
df_sig_ri=df_sig_ri[df_sig_ri['geneSymbol'].isin(DASE_GENE)]
df_sig_ri


,Class,GeneID,geneSymbol,chr,strand,coho4,coho5
19,RI,ENSG00000064601,CTSA,chr20,+,45892323,45892397
21,RI,ENSG00000092330,TINF2,chr14,-,24240330,24240418
22,RI,ENSG00000085998,POMGNT1,chr1,-,46194651,46194843
23,RI,ENSG00000085998,POMGNT1,chr1,-,46194961,46195810
25,RI,ENSG00000239900,ADSL,chr22,+,40361635,40362980
26,RI,ENSG00000077782,FGFR1,chr8,-,38428093,38428351
28,RI,ENSG00000115317,HTRA2,chr2,+,74531702,74531855
29,RI,ENSG00000204843,DCTN1,chr2,-,74368131,74368727
30,RI,ENSG00000118873,RAB3GAP2,chr1,-,220167399,220167501


In [9]:
!ls /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/

GRCh38.primary_assembly.genome.chr_only_clean_2.fa
GRCh38.primary_assembly.genome.chr_only_clean_2.fa.fai
GRCh38.primary_assembly.genome.chr_only.fa
GRCh38.primary_assembly.genome.chr_only.fa.fai
GRCh38.primary_assembly.genome.fa
GRCh38.primary_assembly.genome.fa.fai
Homo_sapiens.GRCh38.112.gtf


In [10]:
#we initialize a position to frame dictionary, encoding the number of nucleotide to skip befor reading the sequence in the correct reading frame.
#the correspondent reading frame will be encoded in the dictionary
position_to_frame={1:3,3:1,2:2}

from the extracted cohordinates we retain 2 nucleotides before the starting of the RI sequence and 2 nucleotides after the end of the RI sequence, so that if a premature stop codon is inserted in the junction proximity between intron and exon, it will be detected.

In [11]:
BED_sig_ri=df_sig_ri[['chr','coho4','coho5','geneSymbol','strand']].copy()
# Insert a column named 'Score' with zeros before the 'strand' column
BED_sig_ri.insert(BED_sig_ri.columns.get_loc('strand'), 'Score', 0)
# Rename columns 'coho1' and 'coho2' to 'Start' and 'End'
BED_sig_ri.rename(columns={'coho4': 'Start', 'coho5': 'End'}, inplace=True)
# Display the updated DataFrame
BED_sig_ri.drop_duplicates(keep='first',inplace=True)
outdir = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons"
os.makedirs(outdir, exist_ok=True)
BED_sig_ri["Start"] = BED_sig_ri["Start"] - 2
BED_sig_ri["End"] = BED_sig_ri["End"] + 2
BED_sig_ri.to_csv('/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/RI_not_expanded_DE_novo_Sig_genome_regions.bed', sep='\t', index=False, header=False)
!bedtools getfasta -fi /home/maria/Desktop/progetto_inquinanti/RMATS_RESULTS/GTF_GRCh38/GRCh38.primary_assembly.genome.fa -bed /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/RI_not_expanded_DE_novo_Sig_genome_regions.bed -fo /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/RI_sig_genome_regions.fa -s -name

In [12]:
BED_sig_ri

,chr,Start,End,geneSymbol,Score,strand
19,chr20,45892321,45892399,CTSA,0,+
21,chr14,24240328,24240420,TINF2,0,-
22,chr1,46194649,46194845,POMGNT1,0,-
23,chr1,46194959,46195812,POMGNT1,0,-
25,chr22,40361633,40362982,ADSL,0,+
26,chr8,38428091,38428353,FGFR1,0,-
28,chr2,74531700,74531857,HTRA2,0,+
29,chr2,74368129,74368729,DCTN1,0,-
30,chr1,220167397,220167503,RAB3GAP2,0,-


In [13]:
def find_premature_stop(sequence, specific_frame=None):
    """
    Checks for premature stop codons in a given sequence with an intron retention variant.

    Args:
        sequence (str): The nucleotide sequence to check (5' to 3').
        specific_frame (int, optional): The specific frame to analyze (1, 2, or 3). If None, analyze all frames.

    Returns:
        list: Information on premature stop codons, containing frame, position, and codon.
    """
    seq = Seq(sequence.upper())
    premature_stops = []

    frames = [specific_frame - 1] if specific_frame else range(3)
    for frame in frames:
        translated_seq = seq[frame:].translate(to_stop=False)
        for i, aa in enumerate(translated_seq):
            if aa == "*":  # Stop codon detected
                stop_position = frame + i * 3 + 1  # Convert to 1-based position
                codon = seq[stop_position - 1:stop_position + 2]
                premature_stops.append({
                    "nucleotides to skip": frame + 1,
                    "stop codon position": stop_position,
                    "codon": str(codon),
                    "reading frame":position_to_frame[frame + 1]
                })
    return premature_stops


In [14]:
def analyze_multifasta_with_tsv(fasta_path, tsv_path):
    """
    Analyze a multi-FASTA file for premature stop codons in specific frames specified in a TSV file.

    Args:
        fasta_path (str): Path to the multi-FASTA file.
        tsv_path (str): Path to the TSV file containing gene names and frames.

    Returns:
        dict: Results for each gene, including premature stop codons for the specified frame.
    """
    # Read the TSV file
    gene_frame_data = pd.read_csv(tsv_path, sep="\t")
    #print(gene_frame_data.head())
    results = {}

    # Parse the FASTA file
    with open(fasta_path, "r") as fasta_file:
        fasta_records = {record.id: str(record.seq) for record in parse(fasta_file, "fasta")}
    #print(fasta_records)
    # Analyze each gene from the TSV
    for _, row in gene_frame_data.iterrows():
        gene = row["gene"]
        position = row["position"]
        
        if gene not in fasta_records:
            #print(gene,frame)
            # Skip if gene not found in the FASTA file
            continue

        sequence = fasta_records[gene]
        #print(sequence)
        strand_match = re.search(r"\((\+|-)\)", gene)
        strand = strand_match.group(1) if strand_match else "+"

        result = find_premature_stop(sequence, specific_frame=position)

        results[gene] = {
            "strand": strand,
            "results": result
        }

    return results

In [15]:
!ls

0-get-real-sequences-from-cohordinates.ipynb
1_premature_stopcodon_identification.ipynb
bAmy-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB118-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_CTR-Tau-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_PCB118-vs-CTR
bAmy-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
bAmy-vs-CTR_AND_PCB153-vs-CTR
CTR-Tau-vs-CTR
CTR-Tau-vs-CTR_AND_PCB118-vs-CTR
CTR-Tau-vs-CTR_AND_PCB118-vs-CTR_AND_PCB153-vs-CTR
CTR-Tau-vs-CTR_AND_PCB153-vs-CTR
DASE_genes_enrichment.ipynb
DEGS_enrichment.ipynb
DEGS_RBP.ipynb
Figure1.ipynb
Goatools.ipynb
mirna-codes
PCB118-vs-CTR
PCB118-vs-CTR_AND_PCB153-vs-CTR
PCB153-vs-CTR


In [16]:
if __name__ == "__main__":
    # File paths
    fasta_file_path = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/RI_sig_genome_regions.fa"  # Replace with your FASTA file path
    tsv_file_path = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/frame_list.tsv"             # Replace with your TSV file path

    results = analyze_multifasta_with_tsv(fasta_file_path, tsv_file_path)

    # Print summarized results
    for gene, result in results.items():
        detected_stops = result["results"]
        if detected_stops:
            count = len(detected_stops)
            frame = detected_stops[0]['reading frame']  # All stops are in the same frame for the gene
            print(f"Results for gene {gene}:")
            print(f"  {count} Premature stop codons detected in Frame {frame}:")
            for stop in detected_stops:
                print(f"    Position: {stop['stop codon position']}, Codon: {stop['codon']}")
        else:
            print(f"Results for gene {gene}:")
            print("  No premature stop codons detected.")

Results for gene CTSA::chr20:45892321-45892399(+):
  1 Premature stop codons detected in Frame 1:
    Position: 6, Codon: TAG
Results for gene TINF2::chr14:24240328-24240420(-):
  1 Premature stop codons detected in Frame 3:
    Position: 4, Codon: TGA
Results for gene POMGNT1::chr1:46194649-46194845(-):
  1 Premature stop codons detected in Frame 2:
    Position: 62, Codon: TAG
Results for gene POMGNT1::chr1:46194959-46195812(-):
  17 Premature stop codons detected in Frame 1:
    Position: 39, Codon: TAG
    Position: 57, Codon: TAG
    Position: 138, Codon: TAA
    Position: 150, Codon: TAA
    Position: 162, Codon: TAG
    Position: 291, Codon: TAA
    Position: 357, Codon: TGA
    Position: 360, Codon: TAG
    Position: 549, Codon: TAG
    Position: 669, Codon: TGA
    Position: 675, Codon: TAG
    Position: 684, Codon: TAA
    Position: 720, Codon: TAA
    Position: 726, Codon: TAG
    Position: 747, Codon: TAA
    Position: 750, Codon: TGA
    Position: 828, Codon: TAA
Results f

/home/maria/anaconda3/lib/python3.9/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [17]:
if __name__ == "__main__":
    # File paths
    fasta_file_path = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/RI_sig_genome_regions.fa"  # Replace with your FASTA file path
    tsv_file_path = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/frame_list.tsv"             # Replace with your TSV file path
    output_file_path = "/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/Premature_stop_codons_in_frame.txt"  # Output file path

    results = analyze_multifasta_with_tsv(fasta_file_path, tsv_file_path)

    # Open the output file in write mode
    with open(output_file_path, "w") as output_file:
        # Write summarized results to the file
        for gene, result in results.items():
            detected_stops = result["results"]
            if detected_stops:
                count = len(detected_stops)
                frame = detected_stops[0]['reading frame']  # All stops are in the same frame for the gene
                output_file.write(f"Results for gene {gene}:\n")
                output_file.write(f"  {count} Premature stop codons detected in Frame {frame}:\n")
                for stop in detected_stops:
                    output_file.write(f"    Position: {stop['stop codon position']}, Codon: {stop['codon']}\n")
            else:
                output_file.write(f"Results for gene {gene}:\n")
                output_file.write("  No premature stop codons detected.\n")

    print(f"Results have been written to {output_file_path}")


Results have been written to /home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/premature_stop_codons/Premature_stop_codons_in_frame.txt


/home/maria/anaconda3/lib/python3.9/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
